
# Consumption–Saving RL — Current Experiment (Fast Setup)

이 노트북은 **현재 실험만** 재현하기 위한 경량 버전입니다.

- 예전 Random / Fixed / DQN v1 / v2 / LR ablation을 다시 실행하지 않습니다.
- `Run all`을 하면 **정의 + DP-Matched benchmark 준비까지만** 빠르게 완료됩니다.
- 맨 마지막 셀의 `RUN_HEAVY_TRAINING = False`를 `True`로 바꾸고 **그 셀만 다시 실행**하면 현재 100k-step sampling ablation이 실행됩니다.
- 런타임 재시작 후에도 처음부터 오래된 실험을 다시 돌릴 필요가 없습니다.

현재 실험:
> v1-long의 모든 조건을 유지하고 **initial-state sampling만 변경**하여 저·중자산 구간의 policy approximation이 개선되는지 확인.


In [ ]:

# ============================================================
# 0. Runtime setup
# ============================================================

import sys
import subprocess
import importlib

# Gymnasium이 없는 경우에만 설치
try:
    import gymnasium as gym
except ModuleNotFoundError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "gymnasium"]
    )
    import gymnasium as gym

# Colab Python 3.13 + 일부 SymPy 1.14 환경에서
# torch.optim 초기화 시 sympy.printing attribute 문제가 생기는 경우를 방지.
import sympy
sympy_printing = importlib.import_module("sympy.printing")

if not hasattr(sympy, "printing"):
    sympy.printing = sympy_printing

assert hasattr(sympy, "printing"), "sympy.printing patch failed"

import random
from collections import deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gymnasium import spaces

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

print("Python:", sys.version.split()[0])
print("SymPy:", sympy.__version__)
print("sympy.printing available:", hasattr(sympy, "printing"))
print("PyTorch:", torch.__version__)
print("Gymnasium:", gym.__version__)


In [ ]:

# ============================================================
# 1. Economic environment + DQN components
# ============================================================

R = 1.00985
BETA = 0.9703
SIGMA = 2.0

Y_HIGH = 1.0
Y_LOW = 0.472

P_INCOME = np.array(
    [
        [0.9393, 0.0607],  # High -> High, Low
        [0.3920, 0.6080],  # Low  -> High, Low
    ],
    dtype=np.float64
)

SAVING_SHARES = np.arange(
    0.0, 1.0, 0.05, dtype=np.float64
)


class ConsumptionSavingEnv(gym.Env):
    """
    Infinite-horizon consumption-saving environment.

    State:
        [asset_t, income_t]

    Action:
        discrete saving-share index
        0 -> 0.00, 1 -> 0.05, ..., 19 -> 0.95

    Reward:
        CRRA utility of current consumption

    Income state:
        0 = High
        1 = Low
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        R=R,
        beta=BETA,
        sigma=SIGMA,
        y_high=Y_HIGH,
        y_low=Y_LOW,
        transition_matrix=P_INCOME,
        initial_asset=1.0,
        initial_income_state=0,
    ):
        super().__init__()

        self.R = float(R)
        self.beta = float(beta)
        self.sigma = float(sigma)

        self.income_values = np.array(
            [y_high, y_low],
            dtype=np.float64
        )

        self.P = np.asarray(
            transition_matrix,
            dtype=np.float64
        )

        if self.P.shape != (2, 2):
            raise ValueError("transition_matrix must be (2,2).")

        if not np.allclose(self.P.sum(axis=1), 1.0):
            raise ValueError("Each row of transition_matrix must sum to 1.")

        self.initial_asset = float(initial_asset)
        self.initial_income_state = int(initial_income_state)

        self.saving_shares = SAVING_SHARES.copy()

        self.action_space = spaces.Discrete(
            len(self.saving_shares)
        )

        self.observation_space = spaces.Box(
            low=np.array([0.0, min(y_low, y_high)], dtype=np.float32),
            high=np.array([np.inf, max(y_low, y_high)], dtype=np.float32),
            dtype=np.float32
        )

        self.asset = None
        self.income_state = None

    def _get_obs(self):
        return np.array(
            [
                self.asset,
                self.income_values[self.income_state]
            ],
            dtype=np.float32
        )

    def _utility(self, consumption):
        if consumption <= 0:
            raise ValueError("Consumption must be positive.")

        if np.isclose(self.sigma, 1.0):
            return float(np.log(consumption))

        return float(
            consumption ** (1.0 - self.sigma)
            / (1.0 - self.sigma)
        )

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)

        options = options or {}

        self.asset = float(
            options.get("initial_asset", self.initial_asset)
        )

        self.income_state = int(
            options.get(
                "initial_income_state",
                self.initial_income_state
            )
        )

        return self._get_obs(), {}

    def step(self, action):
        current_asset = self.asset
        current_income_state = self.income_state
        current_income = self.income_values[current_income_state]

        cash_on_hand = (
            self.R * current_asset
            + current_income
        )

        saving_share = self.saving_shares[action]

        next_asset = (
            saving_share * cash_on_hand
        )

        consumption = (
            cash_on_hand - next_asset
        )

        reward = self._utility(consumption)

        next_income_state = int(
            self.np_random.choice(
                2,
                p=self.P[current_income_state]
            )
        )

        self.asset = next_asset
        self.income_state = next_income_state

        # Infinite-horizon model: no economic terminal state
        terminated = False
        truncated = False

        return (
            self._get_obs(),
            reward,
            terminated,
            truncated,
            {}
        )


def normalize_state(state):
    state = np.asarray(
        state,
        dtype=np.float32
    ).copy()

    state[..., 0] /= 4.5

    return state


class ReplayBuffer:

    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)

    def put(self, state, action, reward, next_state):
        self.buffer.append(
            (
                normalize_state(state),
                int(action),
                float(reward),
                normalize_state(next_state)
            )
        )

    def sample(self, batch_size):
        batch = random.sample(
            self.buffer,
            batch_size
        )

        states, actions, rewards, next_states = zip(*batch)

        return (
            torch.tensor(
                np.asarray(states),
                dtype=torch.float32
            ),
            torch.tensor(
                actions,
                dtype=torch.long
            ),
            torch.tensor(
                rewards,
                dtype=torch.float32
            ),
            torch.tensor(
                np.asarray(next_states),
                dtype=torch.float32
            )
        )

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(2, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 20)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        return self.fc3(x)


print("Core definitions loaded.")


In [ ]:

# ============================================================
# 2. DP-Matched benchmark
# ============================================================

def solve_dp_matched(
    R=R,
    beta=BETA,
    sigma=SIGMA,
    y_high=Y_HIGH,
    y_low=Y_LOW,
    transition_matrix=P_INCOME,
    a_min=0.0,
    a_max=4.5,
    n_assets=500,
    saving_shares=SAVING_SHARES,
    tolerance=1e-10,
    max_iterations=5000
):
    P = np.asarray(
        transition_matrix,
        dtype=np.float64
    )

    income_values = np.array(
        [y_high, y_low],
        dtype=np.float64
    )

    asset_grid = np.linspace(
        a_min,
        a_max,
        n_assets
    )

    V = np.zeros(
        (n_assets, 2),
        dtype=np.float64
    )

    policy_share = np.zeros_like(V)

    for iteration in range(max_iterations):

        V_new = np.empty_like(V)

        for income_state in range(2):

            income = income_values[income_state]

            cash_on_hand = (
                R * asset_grid + income
            )

            Q = np.full(
                (n_assets, len(saving_shares)),
                -np.inf,
                dtype=np.float64
            )

            for j, z in enumerate(saving_shares):

                next_asset = (
                    z * cash_on_hand
                )

                consumption = (
                    (1.0 - z) * cash_on_hand
                )

                feasible = (
                    (consumption > 1e-12)
                    &
                    (next_asset <= a_max)
                )

                V_next_high = np.interp(
                    next_asset,
                    asset_grid,
                    V[:, 0]
                )

                V_next_low = np.interp(
                    next_asset,
                    asset_grid,
                    V[:, 1]
                )

                expected_future = (
                    P[income_state, 0] * V_next_high
                    +
                    P[income_state, 1] * V_next_low
                )

                if np.isclose(sigma, 1.0):
                    utility = np.log(consumption)
                else:
                    utility = (
                        consumption ** (1.0 - sigma)
                        /
                        (1.0 - sigma)
                    )

                q = (
                    utility
                    +
                    beta * expected_future
                )

                q[~feasible] = -np.inf
                Q[:, j] = q

            best_idx = np.argmax(
                Q,
                axis=1
            )

            V_new[:, income_state] = (
                Q[np.arange(n_assets), best_idx]
            )

            policy_share[:, income_state] = (
                saving_shares[best_idx]
            )

        diff = np.max(
            np.abs(V_new - V)
        )

        V = V_new

        if diff < tolerance:
            break

    return asset_grid, V, policy_share


asset_grid_m, V_dp_matched, policy_share_matched = (
    solve_dp_matched()
)

dp_matched_initial_value = np.interp(
    1.0,
    asset_grid_m,
    V_dp_matched[:, 0]
)

low_asset_mask = (
    asset_grid_m <= 0.5
)

middle_asset_mask = (
    (asset_grid_m > 0.5)
    &
    (asset_grid_m <= 2.0)
)

high_asset_mask = (
    asset_grid_m > 2.0
)

print(
    "DP-Matched Bellman value at (asset=1, High):",
    dp_matched_initial_value
)

print("DP-Matched benchmark ready.")


In [ ]:

# ============================================================
# 3. Evaluation helpers
# ============================================================

def utility_vectorized(consumption, sigma=SIGMA):
    if np.isclose(sigma, 1.0):
        return np.log(consumption)

    return (
        consumption ** (1.0 - sigma)
        /
        (1.0 - sigma)
    )


def evaluate_dqn_vectorized(
    q_network,
    n_episodes=2000,
    horizon=500,
    seed=10000,
    R=R,
    beta=BETA,
    sigma=SIGMA,
    y_high=Y_HIGH,
    y_low=Y_LOW
):
    rng = np.random.default_rng(seed)

    assets = np.ones(n_episodes)

    income_states = np.zeros(
        n_episodes,
        dtype=int
    )

    total_utility = np.zeros(n_episodes)

    discount = 1.0

    q_network.eval()

    for _ in range(horizon):

        incomes = np.where(
            income_states == 0,
            y_high,
            y_low
        )

        states = np.column_stack(
            [assets, incomes]
        ).astype(np.float32)

        states[:, 0] /= 4.5

        with torch.no_grad():

            actions = (
                q_network(
                    torch.tensor(
                        states,
                        dtype=torch.float32
                    )
                )
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

        z = SAVING_SHARES[actions]

        cash_on_hand = (
            R * assets + incomes
        )

        consumption = (
            (1.0 - z)
            * cash_on_hand
        )

        reward = utility_vectorized(
            consumption,
            sigma
        )

        total_utility += (
            discount * reward
        )

        assets = (
            z * cash_on_hand
        )

        prob_low = np.where(
            income_states == 0,
            P_INCOME[0, 1],
            P_INCOME[1, 1]
        )

        income_states = (
            rng.random(n_episodes)
            < prob_low
        ).astype(int)

        discount *= beta

    mean = total_utility.mean()
    std = total_utility.std(ddof=1)
    se = std / np.sqrt(n_episodes)
    ci = 1.96 * se

    return {
        "mean_utility": mean,
        "std_utility": std,
        "ci_lower": mean - ci,
        "ci_upper": mean + ci,
        "max_final_asset": assets.max()
    }


def get_dqn_policy_on_grid(
    q_network,
    asset_grid
):
    q_network.eval()

    policies = np.zeros(
        (len(asset_grid), 2)
    )

    income_values = [
        Y_HIGH,
        Y_LOW
    ]

    for income_state, income in enumerate(
        income_values
    ):

        states = np.column_stack(
            [
                asset_grid,
                np.full(
                    len(asset_grid),
                    income
                )
            ]
        ).astype(np.float32)

        states[:, 0] /= 4.5

        with torch.no_grad():

            actions = (
                q_network(
                    torch.tensor(
                        states,
                        dtype=torch.float32
                    )
                )
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

        policies[:, income_state] = (
            SAVING_SHARES[actions]
        )

    return policies


print("Evaluation helpers loaded.")



# Next Experiment — Checkpointed DQN v1-long

이번 실험은 기존 v1-long의 학습 설정을 바꾸지 않고,
5,000 step마다 validation을 수행해서 가장 좋은 checkpoint를 저장합니다.

목적:
- 30k와 100k 성능이 거의 비슷했던 이유가 학습 중 drift/instability 때문인지 확인
- final network와 best checkpoint를 동일 test에서 비교


In [ ]:

def train_dqn_checkpointed(
    total_steps=100_000,
    seed=42,
    learning_rate=1e-3,
    gamma=BETA,
    batch_size=64,
    buffer_size=100_000,
    warmup_steps=1_000,
    target_update_interval=500,
    epsilon_start=1.0,
    epsilon_end=0.05,
    epsilon_decay_steps=25_000,
    reset_interval=200,
    eval_interval=5_000,
    validation_episodes=512,
    validation_horizon=300,
    validation_seed=54321
):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    rng = np.random.default_rng(seed)
    env = ConsumptionSavingEnv()

    q = QNetwork()
    target_q = QNetwork()
    target_q.load_state_dict(q.state_dict())
    target_q.eval()

    optimizer = optim.Adam(q.parameters(), lr=learning_rate)
    replay_buffer = ReplayBuffer(capacity=buffer_size)

    def sample_initial_state():
        return rng.uniform(0.0, 4.5), int(rng.integers(0, 2))

    a0, y0 = sample_initial_state()
    state, _ = env.reset(
        seed=seed,
        options={"initial_asset": a0, "initial_income_state": y0}
    )

    losses = []
    history = []
    episode_step = 0

    best_validation_utility = -np.inf
    best_step = None
    best_state_dict = None

    for step in range(1, total_steps + 1):
        fraction = min((step - 1) / epsilon_decay_steps, 1.0)
        epsilon = epsilon_start + fraction * (epsilon_end - epsilon_start)

        if rng.random() < epsilon:
            action = int(rng.integers(0, 20))
        else:
            s = torch.tensor(
                normalize_state(state),
                dtype=torch.float32
            ).unsqueeze(0)
            with torch.no_grad():
                action = int(q(s).argmax(dim=1).item())

        next_state, reward, _, _, _ = env.step(action)
        replay_buffer.put(state, action, reward, next_state)
        state = next_state
        episode_step += 1

        if episode_step >= reset_interval:
            a0, y0 = sample_initial_state()
            state, _ = env.reset(
                options={"initial_asset": a0, "initial_income_state": y0}
            )
            episode_step = 0

        if len(replay_buffer) >= warmup_steps:
            states, actions, rewards, next_states = replay_buffer.sample(batch_size)

            q_selected = (
                q(states)
                .gather(1, actions.unsqueeze(1))
                .squeeze(1)
            )

            with torch.no_grad():
                max_next_q = target_q(next_states).max(dim=1).values
                target = rewards + gamma * max_next_q

            loss = F.mse_loss(q_selected, target)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(q.parameters(), max_norm=10.0)
            optimizer.step()

            losses.append(loss.item())

        if step % target_update_interval == 0:
            target_q.load_state_dict(q.state_dict())

        if step % eval_interval == 0:
            validation = evaluate_dqn_vectorized(
                q,
                n_episodes=validation_episodes,
                horizon=validation_horizon,
                seed=validation_seed
            )

            policy = get_dqn_policy_on_grid(q, asset_grid_m)
            err = np.abs(policy - policy_share_matched)

            row = {
                "step": step,
                "validation_utility": validation["mean_utility"],
                "policy_mae": err.mean(),
                "exact": np.mean(np.isclose(policy, policy_share_matched)),
                "within5": np.mean(err <= 0.0500001),
                "within10": np.mean(err <= 0.1000001),
                "epsilon": epsilon
            }
            history.append(row)

            if validation["mean_utility"] > best_validation_utility:
                best_validation_utility = validation["mean_utility"]
                best_step = step
                best_state_dict = {
                    k: v.detach().cpu().clone()
                    for k, v in q.state_dict().items()
                }

            print(
                f"step={step:6d} | "
                f"val utility={validation['mean_utility']:.4f} | "
                f"MAE={err.mean():.4f} | "
                f"best={best_step}"
            )

    best_q = QNetwork()
    best_q.load_state_dict(best_state_dict)
    best_q.eval()

    return (
        q,
        best_q,
        target_q,
        losses,
        pd.DataFrame(history),
        best_step
    )



## Heavy training cell

**계산부하: 무거움.**
기존 v1-long 100k보다 validation checkpoint가 추가되어 조금 더 무겁습니다.

`Run all`에서는 아래 셀이 실행되지 않도록 기본값을 `False`로 두었습니다.


In [ ]:

RUN_HEAVY_TRAINING = False

if not RUN_HEAVY_TRAINING:
    print(
        "Fast setup complete.\n"
        "Change RUN_HEAVY_TRAINING = True and rerun THIS CELL ONLY."
    )
else:
    (
        q_final,
        q_best,
        target_q_final,
        losses_checkpointed,
        checkpoint_history,
        best_step
    ) = train_dqn_checkpointed(
        total_steps=100_000,
        seed=42
    )

    print("\nBest validation checkpoint:", best_step)

    final_test = evaluate_dqn_vectorized(
        q_final,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    best_test = evaluate_dqn_vectorized(
        q_best,
        n_episodes=2000,
        horizon=500,
        seed=10000
    )

    final_policy = get_dqn_policy_on_grid(q_final, asset_grid_m)
    best_policy = get_dqn_policy_on_grid(q_best, asset_grid_m)

    final_error = np.abs(final_policy - policy_share_matched)
    best_error = np.abs(best_policy - policy_share_matched)

    results = pd.DataFrame([
        {
            "Model": "Previous v1-long 100k",
            "Mean Utility": -36.68906523595561,
            "Policy MAE": 0.04815,
            "Exact": 0.416,
            "Within 5pp": 0.719,
            "Within 10pp": 0.926
        },
        {
            "Model": "Checkpoint run: final 100k",
            "Mean Utility": final_test["mean_utility"],
            "Policy MAE": final_error.mean(),
            "Exact": np.mean(np.isclose(final_policy, policy_share_matched)),
            "Within 5pp": np.mean(final_error <= 0.0500001),
            "Within 10pp": np.mean(final_error <= 0.1000001)
        },
        {
            "Model": f"Checkpoint run: best @ {best_step}",
            "Mean Utility": best_test["mean_utility"],
            "Policy MAE": best_error.mean(),
            "Exact": np.mean(np.isclose(best_policy, policy_share_matched)),
            "Within 5pp": np.mean(best_error <= 0.0500001),
            "Within 10pp": np.mean(best_error <= 0.1000001)
        }
    ])

    display(results)

    print("\nFinal 95% CI:", (final_test["ci_lower"], final_test["ci_upper"]))
    print("Best 95% CI:", (best_test["ci_lower"], best_test["ci_upper"]))

    region_rows = []
    for model_name, err in [
        ("Final 100k", final_error),
        (f"Best @{best_step}", best_error)
    ]:
        region_rows.append({
            "Model": model_name,
            "Low asset MAE": err[low_asset_mask].mean(),
            "Middle asset MAE": err[middle_asset_mask].mean(),
            "High asset MAE": err[high_asset_mask].mean()
        })

    display(pd.DataFrame(region_rows))

    plt.figure(figsize=(10, 5))
    plt.plot(
        checkpoint_history["step"],
        checkpoint_history["validation_utility"],
        marker="o"
    )
    plt.axvline(
        best_step,
        linestyle="--",
        label=f"Best checkpoint = {best_step}"
    )
    plt.xlabel("Training Step")
    plt.ylabel("Validation Discounted Utility")
    plt.title("DQN Validation Performance During Training")
    plt.legend()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.plot(
        checkpoint_history["step"],
        checkpoint_history["policy_mae"],
        marker="o"
    )
    plt.xlabel("Training Step")
    plt.ylabel("Policy MAE vs DP-Matched")
    plt.title("Policy Approximation Error During Training")
    plt.show()
